In [4]:
import os 
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is set")
else:
    raise ValueError("OPENAI_API_KEY is not set")

OPENAI_API_KEY is set


In [5]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

loader = DirectoryLoader(
    "./IKSPL_Emp_Policies",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)
docs = loader.load()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

In [7]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
print("Embedding test:", len(embeddings.embed_query("test")))

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2246.92it/s]


Embedding test: 384


In [9]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Create LLM
llm = ChatOpenAI(model="gpt-5-mini",temperature=0.9)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant acting as an HR representative of iScholar.
Your role is to answer employee queries strictly based on iScholar's company policies and internal knowledge.

- Answer ONLY using the provided context.
- Do NOT provide generic or external knowledge.
- Ensure responses are specific to iScholar policies.
- Maintain a professional HR tone.

If the answer is not found in the context, respond exactly with: "I could not find this in the current iScholar policies. Please check with HR for clarification.".

Context:
{context}
"""),
    ("human", "{question}")
])

# Step 2: Create QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_template}
)

In [10]:
print(qa_chain.input_keys)

['query']


In [11]:
response = qa_chain.invoke({
    'query': "What is leave policy?"
})
print(response["result"])

iScholar’s Leave & Attendance Policy (summary)

- Objective: To enable employees to maintain a healthy work–life balance. Leave is granted for sickness, recuperation of health, emergencies, personal work, rest and recreation, and fulfilling social obligations.  
- Applicability: The policy applies to all regular, permanent staff of IKSPL.  
- Leave year: Calendar year (January to December).  
- Entitlement: All regular employees are entitled to 24 days of leave per year: Casual Leave (CL) – 6 days, Sick Leave (SL) – 6 days, Earned Leave (EL) – 12 days.  
- New hires: Employees appointed during the year receive leave on a pro‑rata basis.  
- Granting leave: Approval depends on work exigencies and is at the discretion of the Reporting Manager/management.  
- Leave counting rules: For leave calculation, intervening Saturdays, Sundays and Company holidays prefixed, suffixed or in between leave are not counted. However, if the leave is Loss of Pay, Saturdays, Sundays and Company holidays in